# 02. Transformers and Sequence Modelling

Data Science Feedback:
- Random Forest's strength is we don't care that much for the individual trees, we care for the number of estimators. Limit the hyperparam testing for the estimators and play around with the number of estimators. Different dataset sizes may require different numbers.
- Cross-validation doesn't improve the results, it makes them more trustworthy. It slows down train time multiple times, so best to use it on small datasets not always for everything.
- Documenting why we do things is incredibly important! It makes sure we understand why we use certan hyperparams or methods and also documents for historic reasons ours thoughts.
- USE MLFLOW! Show params, hyperparams, explainability, track stuff. Track which dataset variabtion you are using, because if you use 2 different ones for expirementation then you can't really compare their metrics
- You can extract the ~50% probability labeled values that are close to the boundry of decision and see if they have something in common, so we can correct the model in some way to predict thme better. ( or just extract the false positives and exlore their commonality)

Scale your data to avoid this with the loss function:

<img src="Images/valley.png" width=400>

When the data is not scaled some features that are larger in values could seem more important, also the larger values affect the gradient step. Just always normalize/scale!

## Loss function

As long as we can define a loss function for our problem that is differenciable, we can solve any problem and do w/e we want

Usually:
- Regression - mean_squared_error
- Classification - cross-entropy

Custom example:
We can have 1 model, 1 calculation and then:
- 2 neurons for 2 regressions (we have 2 y1 and y2 true labels in the dataset) in the last layer and add them both to the loss function:
- N classification neurons (for each class) with cross entropy

Also, we can add weights to each outcome to the loss function.

All of this in one NN. We can do w/e we want:\
<img src="Images/loss_function_y1_y2_y3.png" width=700>

## 3 pillars of AI (types of architectures)

3 types of architectures that allows us to represent non numeric data as numeric:
- Convolutions - designed to understand images
    - "Local patterns matter." CNNs use "filters" that slide across an image (the convolution) to look for specific features like edges, corners, or textures. Because it looks at small patches at a time, it is excellent at recognizing an object regardless of where it is in the frame. 
- Recurrent - designed to process sequential data (text, speech, time-series)
    - "The past matters." Unlike standard networks that process everything at once, RNNs process data sequentially.
    - They have a "loop" (recurrence) that allows information to persist. As it reads the next piece of data, it remembers what it saw previously. 
- Attension/Transformers - designed to understand complex relationships in large datasets
    - "Focus matters." Instead of processing step-by-step like an RNN, it looks at the entire sequence at once and uses "Self-Attention" to decide which parts are the most important to focus on.
 
Currently 2026.01.31 Attension based models are the most interesting/used

### Model Atchitectures

<img src="Images/model_atchitecture.png" width=700>

- One to one: Standard.
    - Used in models that dont deal with sequences.
- One to many : Sequence generation given seed (image captioning)
- Many to one: One output for sequence (sentiment analysis, forecasting next thing(word for example))
    - Give it a sentence and it gives the next word.
- Many to many:
    - Offset (in the image)
        - The better way to make translation (compared to the non offset method, we will see why): English -> French 
    - Not offset (in the image)
        - Classification of each token. Is the word a country or a geographical place or w/e.
            - <img src="Images/sentence_example.png" width=600>
        - Sentiment analysis - how positive or negative is each word from a review lets say.
    

## Tokenizer



Hugging Face provides access through its Model Hub to thousands of pre-trained models for tasks like speech recognition, text classification, text generation, text summarization, question answering, image generation and more. (Like github but for models)

The tokenizer class gives us access to its models

Lets look at BERT(Bidirectional Encoder Representations from Transformers)

In [33]:
from tokenizers import Tokenizer
from transformers import AutoTokenizer

from keras.models import Sequential
from keras.layers import Embedding, Input, Dense

In [6]:
bert_tokenizer = Tokenizer.from_pretrained("bert-base-uncased")

/home/alpine/miniconda3/envs/dl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
bert_tokenizer.encode("A black cat with a black hat").tokens

['[CLS]', 'a', 'black', 'cat', 'with', 'a', 'black', 'hat', '[SEP]']

In [9]:
bert_tokenizer.encode("A black cat with a black hat").ids

[101, 1037, 2304, 4937, 2007, 1037, 2304, 6045, 102]

In [11]:
bert_tokenizer.id_to_token(2304)

'black'

In [12]:
bert_tokenizer.get_vocab_size()

30522

In [13]:
bert_tokenizer.encode("asd afasf asafagw ").tokens

['[CLS]',
 'as',
 '##d',
 'af',
 '##as',
 '##f',
 'asa',
 '##fa',
 '##g',
 '##w',
 '[SEP]']

We can see what happens when it encounters words it doesn't know

The smallest unit(token) in BERT is a word, which could be a problem in languages that work in a way that letters change depending where they are used.

We can have byte sized tokenizers like BPE(Byte-Pair Encoding): every 2 letters/bytes is a token

In [15]:
bpe_tokenizer = AutoTokenizer.from_pretrained("gpt2")

100%|████████████████████████████████████████████████████████████████████████| 456318/456318 [00:00<00:00, 957329.12B/s]


In [18]:
bpe_tokenizer.vocab_size

50257

A token doesn't need to be human readable. In BPE we cant have invalid values

In [23]:
bpe_tokenizer.encode("A black cat with a black hat")

[32, 2042, 3797, 351, 257, 2042, 6877]

In [21]:
bpe_tokenizer.encode("A BLaCK cat with a black hat")

[32, 9878, 64, 34, 42, 3797, 351, 257, 2042, 6877]

In [24]:
bpe_tokenizer.decode([32, 9878, 64, 34, 42, 3797, 351, 257, 2042, 6877])

'A BLaCK cat with a black hat'

Capital letters are saved here

### Accents

In [26]:
bert_tokenizer.decode(bert_tokenizer.encode("schöne").ids)

'schone'

We can see it becomes a different word. 

What we can do is pre-process and remove these words/accents/diacritics(the 2 dots above words)

If we were to create a sparse matrix for BERT it would be huge:

In [28]:
512 * bert_tokenizer.get_vocab_size()

15627264

512 is the max tokens(max sequence)

15 mil is a lot. Lets look at dim reduction:

### Embedding

(Also known as dimentionality reduction)

At its simplest, an embedding is a way of translating "human-style" data (words, images, products) into a list of numbers (a vector) that a computer can actually do math with.

But it’s not just any list of numbers. An embedding is a dense representation where the geometric distance between points represents the semantic relationship between the objects.

> Essentially we are vectorizing the words (for example) in such a way that the semantics meaning is kept in a geometric way. Similar words are close together.

In [29]:
Embedding(input_dim = bert_tokenizer.get_vocab_size(), output_dim = 1000)

<Embedding name=embedding, built=False>

In [30]:
512 * 1000

512000

30 times less than the sparse matrix before

In [36]:
Sequential([
    Input((512,bert_tokenizer.get_vocab_size())),
    Dense(1000, use_bias=False)
]).summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 512, 1000)      │    30,522,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,522,000 (116.43 MB)

 Trainable params: 30,522,000 (116.43 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
Sequential([
    Input((1,bert_tokenizer.get_vocab_size())),
    Dense(15000, activation="relu"),
    Dense(5000, activation="relu"),
    Dense(1000, use_bias=False)
]).summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 1, 15000)       │   457,845,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1, 5000)        │    75,005,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1, 1000)        │     5,000,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 537,850,000 (2.00 GB)

 Trainable params: 537,850,000 (2.00 GB)

 Non-trainable params: 0 (0.00 B)

What we did:
- Divided the text into tokens(numbers)
- Treated them as categorical values and one-hot encoded them. Saw that that dimentionality is too big
- So we used dim reduction (Embedding) to reduce it
- Output is a smaller vector 1k, just a typical NN output vector, it is not the same encoding as the input

The "magic" (semantics being kept geometrically close) happens because the neural network is forced to solve a riddle: "Given these words, what word is likely to come next?"

To solve that riddle, it has to learn that "Coffee" and "Tea" are similar, otherwise, it would have to learn two completely different sets of rules for every hot beverage.

### From Raw Text to Semantic Vectors: The 3-Step Process (Gemini generated example, that explains it pretty well)

#### 1. Tokenization (The Identity Phase)
Computers cannot process "strings" of text directly. The first step is to map every unique word in your vocabulary to a specific integer ID.
* **Process:** The text is split into pieces (tokens) and assigned a number from a dictionary.
* **Example:** `"The King rules"` → `{"The": 1, "King": 2, "rules": 3}`
* **Limitation:** This step has **zero** semantic awareness. The computer doesn't know that "King" and "Queen" are related yet; they are just arbitrary numbers.



#### 2. The Embedding Layer (The Lookup Table)
The Embedding Layer is a massive, trainable **Weight Matrix** that acts as the interface between the discrete ID and the neural network.
* **Rows:** Represent the vocabulary size (e.g., 10,000 words).
* **Columns:** Represent the "embedding dimensions" (e.g., 128 or 300).
* **Action:** When you pass in a Token ID, the layer performs a **lookup**. If you pass in `ID 2`, it retrieves the 128-number vector stored in the 2nd row of that matrix. These numbers are the "coordinates" of the word in high-dimensional space.



#### 3. Learning Semantics (The Contextual Nudge)
Semantics are a byproduct of training. The model is usually trained on a "Context Task," like predicting a missing word in a sentence.
* **The Logic:** If "King" and "Queen" frequently appear in similar sentences (e.g., "The ___ sat on the throne"), they are interchangeable in the eyes of the model.
* **The Optimizer's Job:** To minimize loss, the optimizer (like Adam) nudges the values in the embedding vectors. It moves "King" and "Queen" closer together in the vector space and moves unrelated words like "Apple" further away.
* **Result:** After training, words with similar meanings end up with similar **angles** (high cosine similarity) in the vector space.



---

> **Note:** We use **Dimensionality Reduction** (e.g., 300 dimensions instead of 50,000) to force the model to learn the most "dense" and meaningful patterns of language rather than just memorizing every word.

### Word2Vec and GloVe (Global Vectors for Word Representation (Embeddings))

Are algorithms that are specifically trained for embeddings.

#### Word2Vec

You get pairs of words.
The context of a word is 3 words before it and 3 words after it.

<img src="Images/word2vec.png" width=500>

This means we won't be looking at every possible pair's relations, just the pairs around it (the 3 interval context around the current word)\
This way the model understands the context of the words, if they fit with each other

##### Word2Vec vs. Transformers: Comparison Table


| Feature | Word2Vec | Transformers (BERT, GPT, etc.) |
| :--- | :--- | :--- |
| **Embedding Type** | **Static**: One fixed vector per word. | **Dynamic**: Vectors change based on context. |
| **Context Range** | **Local**: Only looks at a small "window" of nearby words. | **Global**: Uses Self-Attention to look at the *entire* sequence. |
| **Core Mechanism** | Shallow Neural Network (Skip-gram/CBOW). | Multi-Head Self-Attention layers. |
| **Meanings** | Struggles with polysemy (e.g., "bank" is always the same). | Excels at disambiguation (distinguishes "river bank" vs. "money bank"). |
| **Math Logic** | Linear relationships ($King - Man + Woman \approx Queen$). | Non-linear, high-dimensional contextual relationships. |
| **Compute** | Low; runs fast on a standard CPU. | High; requires GPUs/TPUs for efficient processing. |

The cool thing about word2Vec is that it generates vectors from words which then keep the semantics. You can see the man->woman and king->queen relation are very similar
If we calculate the (man,woman) vector and construct it from king's start we should end up around queen, then we can look at that proximity range and find the nearest word, which would be queen probably

<img src="Images/word2vec_relations.png" width=600>

Cosine similarity is often enough to understand how close words are, even if their euclidian distance is big (the length of the vecor might be different, but the angle mattters more. The length doesn't affect semantics)

<img src="Images/cos_similarity.png" width=600>

* The embedding and its dimentionality reduction transformes the vectors from one hot encoded(a lot of 0 and only one 1)(0, 0, 0, 1, 0, 0 ... 0) to less columns that are floating point values (0.2, 0.5, 0,1 ...)

Ok so we have these floating point vectors that are encoded words, what the hell do we do with it?

### Convolution

Smoothing a tensor(function, array, matrix w/e) with a kernal kernel. Means mixing.

<img src="Images/convolution_1d.png" width=600>

<img src="Images/convolution.gif" width=600>

Convolutions are a way to represent time series

## [Attension](https://sebastianraschka.com/blog/2023/self-attention-from-scratch.html)

Attenstion comes from search engines.


We have 3 things:
- Query - Think of this as you walking around with a key
    - Like an SQL query ``` SELECT hair FROM people ```
- Key - Think of this as a keyhole that could fit the key in your hand
    - Comes from key value pair, like a dictionary ``` {"hair": "blue}```
    - the key: "hair"
- Value - Think of this as the meaning behind the door
    - The value of the key value pair
    - the "blue

In reality all these Query, Key, Value are matrixes that are (word_dictionary.count() x vector_length), where vector_length is the size of the dim reduction of the tokenized words from the dictionary of all words.\
(300 long vector for each of 30,000 words/tokens from the tokenizer, so 300x30,000 matrixes)

So we have a vector_length long vectors for each word in the matrixes representing that word's:
- query
- key
- value

Since all the vectors have the same length (300 in our case) we can compare them with cosine distance. This is how we "compare" the query and key (key in your hand and the keyhole) of one word and anothers.

Now we can multiply each word's query (key in your hand) to every other word's key (keyhole) and get a scalar value and this way we know their relationship. Big numbers means they are similar. This determines how close is the thought of word one with the thought of word 2.

<img src="Images/query_key_heatmap.png" width=500>

Here we can see the $ Q * K^T $ heatmap. It is a matrix of the relationships with one word's query and another word's key. Notice they are not symetrical on the diagonal. 

> A cat wants to sit, but a sit doesn't want to cat.

Genius example I know ...

Once we have that heatmap we normalize it ( $ \sqrt{d_k} $ ) and hit it with a softmax:

$$ A(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

This forces the matrix of scalars to become a matrix of probabilities, then we multiply by V, this way the value is not affected by the softmax and Q and K.

This allows the loss function to tweak the Q K and V differently (because of the way they affect the outcome), which gives them the meaning we assign to them. Otherwise to the loss function they are just matrixes with numbers, not Query, Key, Value, just numbers.

#### Why V can be different dimensions (Gemini)

<img src="Images/V_length_explanation.png" width=500>

> Think of V: If the Q*K is the match that happens when we search for a YuTube video's title, when we multiply the softmaxxed(Q*K) with V, we get the link to that matches video. Kind of like projection.

#### Finally how we make the choice (Gemini)

We multiply the $ A(Q, K, V) $ to the initial dictionary vector space to see which word (from the dictionary) "matches" (dot product) with what we need to output as a result and choose that word with another softmax

<img src="Images/V_out_choice.png" width=500>

Another way to think about attension: "Think of a convolution which's kernal is as big as the whole signal/image/data. A bit more refined but that is attension. You make it look at the whole picture, not just a small sliding window."

#### Sparse Attention (optimize memory)

Attension looks at all the pairs of tokens, that is often inefficient and too much. So we optimized it in this way:
- We add global pairs
- We add local pairs
- A bit of randomness for some spice

<img src="Images/attension_optimization.png" width=500>


#### [Flash Attention](https://arxiv.org/abs/2205.14135) (optimize speed)

Used to optimize the number of operations on a single gpu. Not memory, number of operations

## Transformers

GPT (Generative Pre-trained Transformer)

Types:
- Transformer Encoders
- Transformer Decoders

- Attension takes ALOT of memory (looks at every pair in the data) and yet it keeps not solving a major problem we have: ()
- Takes too long to train 


Lets look at GPT-2

In [2]:
from transformers import GPT2LMHeadModel, AutoConfig

In [3]:
config = AutoConfig.from_pretrained("gpt2")
config

100%|█████████████████████████████████████████████████████████████████████████████| 665/665 [00:00<00:00, 1495556.12B/s]


{
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "finetuning_task": null,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_layer": 12,
  "n_positions": 1024,
  "num_labels": 1,
  "output_attentions": false,
  "output_hidden_states": false,
  "output_past": true,
  "pruned_heads": {},
  "resid_pdrop": 0.1,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "torchscript": false,
  "use_bfloat16": false,
  "vocab_size": 50257
}

In [4]:
gpt2 = GPT2LMHeadModel(config)

In [6]:
gpt2

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [10]:
gpt2.transformer.h[0].attn.c_attn.weight

Parameter containing:
tensor([[ 0.0160, -0.0124, -0.0295,  ...,  0.0076, -0.0420, -0.0014],
        [-0.0039,  0.0068, -0.0064,  ..., -0.0044, -0.0173, -0.0174],
        [ 0.0118, -0.0191,  0.0030,  ..., -0.0107, -0.0172, -0.0123],
        ...,
        [ 0.0235, -0.0291, -0.0085,  ...,  0.0234,  0.0207,  0.0216],
        [-0.0182,  0.0129, -0.0063,  ...,  0.0086,  0.0057,  0.0170],
        [ 0.0032,  0.0082, -0.0076,  ..., -0.0188, -0.0098,  0.0591]],
       requires_grad=True)

### Encoder/Decoder Transformer

The red part we call ENCODER, the green part DECODER. We reduce or increase the dimensions

<img src="Images/encoder_decoder.png" width=800>

Where are the Query Key Value matrixes then? What is that Conv1D?

<img src="Images/missing_Q_K_V.png" width=800>

### Positional Embedding

A problem transformers have is: they miss the order of the words (or patches of pixels in images, even worse there cus its 2d we have, up, down, left and right order, not just left and right like in text)

This is why we have to give it a position embedding so it understands the order as well.

- Classically done using circular encoding (sine / cosine)
- Recentry, with just another layer passed as input
    - "A black cat in a black hat."
    - [0, 1, 2, 3, 4, 5, 6, 7]
  
<img src="Images/pos_emb.png" width=500>

<img src="Images/positional_embedding.png" width=500>

- wte: Word token embedding
- wpe: Word positional embedding
    - This is where we give it the positinal information 

### Pre-training

Pre-training is the initial phase of training an LLM where it learns from a large, diverse dataset of often trillions of tokens. The goal here is to develop a broad understanding of language, context, and various types of knowledge. Pre-training is usually MASSIVELY computationally expensive and requires HUGE amounts of data. We're often talking in the millions of dollars when pre-training models. This is done once for every known LLM, since it is so expensive.

### Fine-tuning

Fine-tuning is where you take an already-pre-trained model and further train it on a more specific dataset. This dataset is typically smaller and focused on a particular domain or task. The purpose of fine-tuning is to adapt the model to perform better in specific scenarios or on tasks that were not well covered during pre-training. The new knowledge added during fine-tuning is more about enhancing the model's performance in specific contexts rather than broadly expanding its general knowledge.

We usually decapitate the old head of the model (last layer) and give it a new head for the specific new task.

### Post-training

Usually done after the fine-tuning with human reinforcement learning. 

Example: already in use as a chatbot but some outputs are better than other for people so we ask them to label which is better and further fine-tune on human feedback to make it better. 
Example: Adhear to company policies like don't tell people how to make bombs and things like that for security purposes.

## Recurrent Neural Networks

They form a loop that passes a context vector with each token that gets changed on each pass through the layer. Then it gets fed back into the same layer with the next token and on and on. 

This recurrence(going back) is why they are called Recurrent Neaural Networks

<img src="Images/rnn_cell.png" width=600>

The 2 omegas in the image are doing the same thing as attention. The first omega is Q * K and the second omega is V 

These cells can be stacked on top of each other

<img src="Images/recurrent_nn.png" width=600>

Backpropagation through time:\
A single layer can be seen as a deep NN becasue of  that recurring loop.

<img src="Images/backprop_through_time.png" width=600>


This context usually runs in exploading/vanishing gradients.

This is a problem that occurs when the derivative chain rules is applied a bit too many times and the gradients reach 0 or infinity and the precision in the memory is lost.

This is fixed by resetting it every a couple of layers.

Such networks are called:
#### Gated RNNs

1. Long Short-Term Memory (LSTM)
2. Gated Recurrent Unit (GRU)

What is modern:

There are a lot of combinations between Convolutions and Attension, but we start seeing a lot of combinations between Gates and Attension

Whatever sticks works. Use if useful.